# 01 — Build the 100 m data foundation

This notebook builds the reproducible 100 m data foundation for the thesis analysis in Styria. It uses the observed 2019 and 2025 POPREG grids as separate small-scale population anchors while leaving the existing firm input unchanged.

## Inputs and Outputs

Required inputs:

- `OGD/Gemeindegrenzen.zip`: municipality boundaries for Styria.
- `OGD/Popreg_100m/population_grid_styria_2019.geoparquet`: prepared 2019 100 m population grid.
- `OGD/Popreg_100m/population_grid_styria_2025.geoparquet`: prepared 2025 100 m population grid.
- `SDG/companies_styria_syn.geoparquet`: current firm point input for pipeline development.
- `OGD/STMK_POP_2002_2025.csv`: annual official municipality populations used for the dual-anchor backcast.

Outputs written by this notebook:

- `ANAL/data/raster_100m_styria.geoparquet`
- `ANAL/data/firms_assigned_100m.geoparquet`
- `ANAL/data/population_backcast_100m_quarterly.parquet`
- `ANAL/data/raster_quarter_panel_100m.parquet`
- `ANAL/data/births_by_fachgruppe_100m.parquet`

## Methodological Notes

The full 100 m raster universe is generated from the Styria municipality boundaries in EPSG:3035. Every grid cell receives a stable `grid_id`, a municipality assignment, and both observed anchor values (`population_2019` and `population_2025`).

The population backcast has two spatial branches. Years 2015–2019 use the observed 2019 cell distribution and annual municipality scaling; years 2020–2025 use the observed 2025 cell distribution and annual municipality scaling. A diagnostic also computes the 2025-anchored estimate for 2019 so the overlap boundary can be compared, but the panel uses the observed 2019 branch in 2019. Annual values are repeated across quarters without implying quarterly precision.

The raster-quarter panel retains the union of cells populated in either observed anchor year and cells with a firm present during the analysis window. Consequently, a cell remains in every panel quarter if it had population in 2019, has population in 2025, or hosted a firm during the window—even when its selected annual population is zero in a particular branch.

`active_firms_t` is descriptive. Model-ready exposure is `active_firms_tminus1` (and the analogous Fachgruppe columns), measured before births in quarter t.

## 1. Imports and Paths

In [1]:
from pathlib import Path
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import box

def discover_project_dir() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "ANAL").is_dir() and (candidate / "OGD").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from the project directory or one of its subdirectories.")

PROJECT_DIR = discover_project_dir()
OGD_DIR = PROJECT_DIR / "OGD"
SDG_DIR = PROJECT_DIR / "SDG"
ANAL_DIR = PROJECT_DIR / "ANAL"
OUTPUT_DIR = ANAL_DIR / "data"

MUNICIPALITIES_PATH = OGD_DIR / "Gemeindegrenzen.zip"
POPULATION_PATHS = {
    2019: OGD_DIR / "Popreg_100m" / "population_grid_styria_2019.geoparquet",
    2025: OGD_DIR / "Popreg_100m" / "population_grid_styria_2025.geoparquet",
}
FIRMS_INPUT_PATH = SDG_DIR / "companies_styria_syn.geoparquet"
MUNICIPAL_POPULATION_PATH = OGD_DIR / "STMK_POP_2002_2025.csv"

RASTER_OUTPUT = OUTPUT_DIR / "raster_100m_styria.geoparquet"
FIRMS_OUTPUT = OUTPUT_DIR / "firms_assigned_100m.geoparquet"
POPULATION_BACKCAST_OUTPUT = OUTPUT_DIR / "population_backcast_100m_quarterly.parquet"
PANEL_OUTPUT = OUTPUT_DIR / "raster_quarter_panel_100m.parquet"
BIRTHS_BY_FACHGRUPPE_OUTPUT = OUTPUT_DIR / "births_by_fachgruppe_100m.parquet"

CRS = "EPSG:3035"
CELL_SIZE = 100
START_YEAR = 2015
END_YEAR = 2025
CENSORING_DATE = pd.Timestamp("2025-12-31")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
MUNICIPALITIES_PATH

WindowsPath('D:/CO2_Masterarbeit/CO2_Masterarbeit/OGD/Gemeindegrenzen.zip')

## 2. Helper Functions

In [3]:
def read_zipped_shapefile(zip_path: Path) -> gpd.GeoDataFrame:
    """Read a zipped shapefile reliably on Windows and Linux."""
    archive_path = zip_path.resolve().as_posix()
    zip_uri = f"zip://{archive_path}"
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message=".*FLAECHE_HA parsed incompletely.*")
        return gpd.read_file(zip_uri)


def make_grid_id(easting: pd.Series, northing: pd.Series) -> pd.Series:
    return "AT_CRS3035RES100mN" + northing.astype("int64").astype(str) + "E" + easting.astype("int64").astype(str)


def date_to_quarter(value: pd.Series) -> pd.Series:
    dates = pd.to_datetime(value, errors="coerce")
    return dates.dt.to_period("Q").astype("string")


def check(condition: bool, message: str) -> None:
    status = "OK" if condition else "CHECK"
    print(f"{status}: {message}")

## 3. Load Existing Inputs

Both population anchors are inspected before derived products are created. The firm input file remains read-only.

In [4]:
municipalities = read_zipped_shapefile(MUNICIPALITIES_PATH).to_crs(CRS)
municipalities = municipalities[["GEMNR6", "GEMNR", "GEMNAM", "geometry"]].copy()
municipalities = municipalities.rename(
    columns={
        "GEMNR6": "municipality_id",
        "GEMNR": "municipality_short_id",
        "GEMNAM": "municipality_name",
    }
)
municipalities["municipality_id"] = municipalities["municipality_id"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True).str.zfill(5)

population_grids = {}
for year, path in POPULATION_PATHS.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing prepared {year} population grid: {path}. Run OGD/Popreg_100m/Population_Raster_preparation.ipynb first.")
    layer = gpd.read_parquet(path).to_crs(CRS)
    required = {"cell_id", "population", "easting", "northing", "cell_size_m", "geometry"}
    missing = required - set(layer.columns)
    if missing:
        raise ValueError(f"{year} population grid lacks columns: {sorted(missing)}")
    if not layer["cell_id"].is_unique:
        raise ValueError(f"{year} population grid contains duplicate cell IDs")
    layer["population"] = pd.to_numeric(layer["population"], errors="raise").fillna(0).astype(float)
    population_grids[year] = layer

population_2019 = population_grids[2019]
population_2025 = population_grids[2025]

firms_source = gpd.read_parquet(FIRMS_INPUT_PATH).to_crs(CRS)
if "Fachgruppe_ID" not in firms_source.columns:
    raise ValueError("Firm input must contain Fachgruppe_ID for the H2b/H2c same-sector measures.")
FACHGRUPPE_IDS = sorted(firms_source["Fachgruppe_ID"].dropna().astype("string").unique().tolist())
if len(FACHGRUPPE_IDS) != 95:
    raise ValueError(f"Expected 95 Fachorganisationen, found {len(FACHGRUPPE_IDS)}.")

print(f"Municipalities: {len(municipalities):,}")
for year, layer in population_grids.items():
    print(f"{year} populated cells: {len(layer):,}; grid population: {layer['population'].sum():,.0f}; CRS: {layer.crs}")
print(f"Firm input rows: {len(firms_source):,}")
print(f"Municipality CRS: {municipalities.crs}")
print(f"Firm CRS: {firms_source.crs}")

Municipalities: 285
2019 populated cells: 121,992; grid population: 1,243,846; CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "ETRS89-extended / LAEA Europe", "base_crs": {"name": "ETRS89", "datum_ensemble": {"name": "European Terrestrial Reference System 1989 ensemble", "members": [{"name": "European Terrestrial Reference Frame 1989"}, {"name": "European Terrestrial Reference Frame 1990"}, {"name": "European Terrestrial Reference Frame 1991"}, {"name": "European Terrestrial Reference Frame 1992"}, {"name": "European Terrestrial Reference Frame 1993"}, {"name": "European Terrestrial Reference Frame 1994"}, {"name": "European Terrestrial Reference Frame 1996"}, {"name": "European Terrestrial Reference Frame 1997"}, {"name": "European Terrestrial Reference Frame 2000"}, {"name": "European Terrestrial Reference Frame 2005"}, {"name": "European Terrestrial Reference Frame 2014"}, {"name": "European Terrestrial Reference Frame 2020"}],

## 4. Build the Full 100 m Raster Universe

The raster is created from the total Styria boundary. It includes zero-population cells so later analyses can define active cells consistently.

In [5]:
minx, miny, maxx, maxy = municipalities.total_bounds

start_x = int(np.floor(minx / CELL_SIZE) * CELL_SIZE)
end_x = int(np.ceil(maxx / CELL_SIZE) * CELL_SIZE)
start_y = int(np.floor(miny / CELL_SIZE) * CELL_SIZE)
end_y = int(np.ceil(maxy / CELL_SIZE) * CELL_SIZE)

eastings = np.arange(start_x, end_x, CELL_SIZE)
northings = np.arange(start_y, end_y, CELL_SIZE)
xx, yy = np.meshgrid(eastings, northings)

candidate_cells = pd.DataFrame(
    {
        "easting": xx.ravel(),
        "northing": yy.ravel(),
    }
)
candidate_cells["centroid_x"] = candidate_cells["easting"] + CELL_SIZE / 2
candidate_cells["centroid_y"] = candidate_cells["northing"] + CELL_SIZE / 2

candidate_points = gpd.GeoDataFrame(
    candidate_cells,
    geometry=gpd.points_from_xy(candidate_cells["centroid_x"], candidate_cells["centroid_y"]),
    crs=CRS,
)

inside_styria = gpd.sjoin(
    candidate_points,
    municipalities[["municipality_id", "geometry"]],
    how="inner",
    predicate="intersects",
).drop(columns=["index_right", "municipality_id"])

inside_styria = inside_styria.drop_duplicates(subset=["easting", "northing"]).reset_index(drop=True)
inside_styria["geometry"] = [
    box(easting, northing, easting + CELL_SIZE, northing + CELL_SIZE)
    for easting, northing in zip(inside_styria["easting"], inside_styria["northing"])
]

raster = gpd.GeoDataFrame(inside_styria, geometry="geometry", crs=CRS)
raster["grid_id"] = make_grid_id(raster["easting"], raster["northing"])

# The source preparation follows the established polygon-intersection clip.
# Add its rare border cells whose centres lie just outside Styria so no cell
# populated in either observed year can disappear from the raster universe.
anchor_cells = pd.concat(
    [layer[["cell_id", "easting", "northing"]] for layer in population_grids.values()],
    ignore_index=True,
).drop_duplicates("cell_id")
missing_anchor_cells = anchor_cells.loc[~anchor_cells["cell_id"].isin(raster["grid_id"])].copy()
if not missing_anchor_cells.empty:
    missing_anchor_cells["centroid_x"] = missing_anchor_cells["easting"] + CELL_SIZE / 2
    missing_anchor_cells["centroid_y"] = missing_anchor_cells["northing"] + CELL_SIZE / 2
    missing_anchor_cells["geometry"] = [
        box(easting, northing, easting + CELL_SIZE, northing + CELL_SIZE)
        for easting, northing in zip(missing_anchor_cells["easting"], missing_anchor_cells["northing"])
    ]
    missing_anchor_cells = missing_anchor_cells.rename(columns={"cell_id": "grid_id"})
    raster = gpd.GeoDataFrame(
        pd.concat([raster, missing_anchor_cells[raster.columns]], ignore_index=True),
        geometry="geometry",
        crs=CRS,
    )
raster["resolution_m"] = CELL_SIZE

print(f"Candidate 100 m cells in bounding box: {len(candidate_cells):,}")
print(f"Centroid-selected Styria raster cells: {len(inside_styria):,}")
print(f"Additional populated boundary-touching cells from either anchor: {len(missing_anchor_cells):,}")
print(f"Final 100 m raster universe: {len(raster):,}")
display(raster.head())

Candidate 100 m cells in bounding box: 2,671,272
Centroid-selected Styria raster cells: 1,641,287
Additional populated boundary-touching cells from either anchor: 160
Final 100 m raster universe: 1,641,447


,easting,northing,centroid_x,centroid_y,geometry,grid_id,resolution_m
0,4740200,2626400,4740250.0,2626450.0,"POLYGON ((4740300 2626400, 4740300 2626500, 47...",AT_CRS3035RES100mN2626400E4740200,100
1,4739900,2626500,4739950.0,2626550.0,"POLYGON ((4740000 2626500, 4740000 2626600, 47...",AT_CRS3035RES100mN2626500E4739900,100
2,4740000,2626500,4740050.0,2626550.0,"POLYGON ((4740100 2626500, 4740100 2626600, 47...",AT_CRS3035RES100mN2626500E4740000,100
3,4740100,2626500,4740150.0,2626550.0,"POLYGON ((4740200 2626500, 4740200 2626600, 47...",AT_CRS3035RES100mN2626500E4740100,100
4,4740200,2626500,4740250.0,2626550.0,"POLYGON ((4740300 2626500, 4740300 2626600, 47...",AT_CRS3035RES100mN2626500E4740200,100


## 5. Assign Municipalities

Centroid assignment is used first because it is simple and reproducible. If a border cell centroid is outside all municipality polygons, the fallback assigns the cell to the municipality with the largest intersecting area.

In [6]:
centroids = raster[["grid_id", "geometry"]].copy()
centroids["geometry"] = centroids.geometry.centroid

centroid_join = gpd.sjoin(
    centroids,
    municipalities[["municipality_id", "municipality_name", "geometry"]],
    how="left",
    predicate="within",
).drop(columns=["index_right"])

raster = raster.merge(
    centroid_join[["grid_id", "municipality_id", "municipality_name"]],
    on="grid_id",
    how="left",
)

missing_municipality = raster["municipality_id"].isna()
print(f"Cells without centroid municipality assignment: {missing_municipality.sum():,}")

if missing_municipality.any():
    unresolved = raster.loc[missing_municipality, ["grid_id", "geometry"]].copy()
    intersections = gpd.overlay(
        unresolved,
        municipalities[["municipality_id", "municipality_name", "geometry"]],
        how="intersection",
    )
    intersections["intersection_area"] = intersections.geometry.area
    fallback = intersections.sort_values("intersection_area", ascending=False).drop_duplicates("grid_id")
    fallback = fallback[["grid_id", "municipality_id", "municipality_name"]]
    raster = raster.drop(columns=["municipality_id", "municipality_name"]).merge(
        pd.concat(
            [
                centroid_join.dropna(subset=["municipality_id"])[["grid_id", "municipality_id", "municipality_name"]],
                fallback,
            ],
            ignore_index=True,
        ),
        on="grid_id",
        how="left",
    )

print(f"Cells still without municipality assignment: {raster['municipality_id'].isna().sum():,}")

Cells without centroid municipality assignment: 160


Cells still without municipality assignment: 0


## 6. Attach the 2019 and 2025 Population Anchors

Each Styria-filtered POPREG layer contains only populated 100 m cells. Both layers are joined independently to the full raster and missing values are zero-filled. Keeping both columns makes population appearance and disappearance explicit and supports the union-based panel-cell rule.

In [7]:
for year, layer in population_grids.items():
    population_table = layer[["cell_id", "population"]].rename(
        columns={"cell_id": "grid_id", "population": f"population_{year}"}
    )
    raster = raster.merge(population_table, on="grid_id", how="left", validate="one_to_one")
    raster[f"population_{year}"] = raster[f"population_{year}"].fillna(0).astype(float)

raster = raster[
    [
        "grid_id",
        "easting",
        "northing",
        "centroid_x",
        "centroid_y",
        "municipality_id",
        "municipality_name",
        "resolution_m",
        "population_2019",
        "population_2025",
        "geometry",
    ]
]

raster.to_parquet(RASTER_OUTPUT, index=False)
print(f"Saved raster universe: {RASTER_OUTPUT}")
for year in POPULATION_PATHS:
    print(f"Total {year} population joined to full raster: {raster[f'population_{year}'].sum():,.0f}")

Saved raster universe: D:\CO2_Masterarbeit\CO2_Masterarbeit\ANAL\data\raster_100m_styria.geoparquet
Total 2019 population joined to full raster: 1,243,846
Total 2025 population joined to full raster: 1,272,478


## 7. Assign Firms to 100 m Cells

The source firm dataset is not changed. This section creates an analytical copy with raster IDs and quarter variables.

In [8]:
firms = firms_source.copy()
firms["firm_id"] = np.arange(1, len(firms) + 1)
firms["founding_date"] = pd.to_datetime(firms["Mitglied_Gründungsdatum"], errors="coerce")
firms["exit_date"] = pd.to_datetime(firms["Mitglied_Löschdatum"], errors="coerce")
firms["founding_quarter"] = date_to_quarter(firms["founding_date"])
firms["exit_quarter"] = date_to_quarter(firms["exit_date"])
firms["exit_observed"] = firms["exit_date"].notna()

firm_join = gpd.sjoin(
    firms[["firm_id", "geometry"]],
    raster[["grid_id", "municipality_id", "municipality_name", "geometry"]],
    how="left",
    predicate="within",
).drop(columns=["index_right"])

firms = firms.merge(
    firm_join[["firm_id", "grid_id", "municipality_id", "municipality_name"]],
    on="firm_id",
    how="left",
)
firms = firms.rename(columns={"grid_id": "grid_id_100m"})

firms.to_parquet(FIRMS_OUTPUT, index=False)
print(f"Saved firm assignment: {FIRMS_OUTPUT}")
print(f"Firms without 100 m grid assignment: {firms['grid_id_100m'].isna().sum():,}")
display(firms.head())

Saved firm assignment: D:\CO2_Masterarbeit\CO2_Masterarbeit\ANAL\data\firms_assigned_100m.geoparquet
Firms without 100 m grid assignment: 0


,Sparte_ID,Sparte_Text,Fachgruppe_ID,Fachgruppe_Text,Mitglied_Gründungsdatum,Standort_Angelegt,Standort_Gelöscht,Mitglied_Löschdatum,OSM_Building_ID,OSM_Building_Type,...,geometry,firm_id,founding_date,exit_date,founding_quarter,exit_quarter,exit_observed,grid_id_100m,municipality_id,municipality_name
0,1,Gewerbe und Handwerk,128,FG Persönliche Dienstleister,1963-03-26,1973-07-09,NaT,NaT,391722827,yes,...,POINT (4746259.779 2695046.029),1,1963-03-26,NaT,1963Q1,<NA>,False,AT_CRS3035RES100mN2695000E4746200,61766,Weiz
1,1,Gewerbe und Handwerk,124,LI Friseure,1977-06-25,2002-01-08,NaT,NaT,334682208,yes,...,POINT (4748507.322 2650071.709),2,1977-06-25,NaT,1977Q2,<NA>,False,AT_CRS3035RES100mN2650000E4748500,61008,Gabersdorf
2,6,Tourismus und Freizeitwirtschaft,606,FG Freizeit- und Sportbetriebe,1957-07-30,1991-04-09,NaT,NaT,406245160,yes,...,POINT (4764583.347 2693758.785),3,1957-07-30,NaT,1957Q3,<NA>,False,AT_CRS3035RES100mN2693700E4764500,62266,Feistritztal
3,1,Gewerbe und Handwerk,119,LI Lebensmittelgewerbe,1973-02-17,1984-01-19,2008-10-16,2022-05-17,355497855,yes,...,POINT (4607515.176 2707186.655),4,1973-02-17,2022-05-17,1973Q1,2022Q2,True,AT_CRS3035RES100mN2707100E4607500,61217,Haus
4,1,Gewerbe und Handwerk,126,FG Gewerbliche Dienstleister,1986-06-28,2021-06-22,2021-07-19,NaT,142166061,yes,...,POINT (4748672.642 2683411.335),5,1986-06-28,NaT,1986Q2,<NA>,False,AT_CRS3035RES100mN2683400E4748600,60661,Eggersdorf bei Graz


## 8. Build the Dual-Anchor Population Backcast

The small-scale distribution is allowed to change between the two observed POPREG grids:

- 2015–2019: scale each 2019 cell within its municipality to the official municipality total for the target year.
- 2020–2025: scale each 2025 cell within its municipality to the official municipality total for the target year.

For cell `i`, municipality `g`, target year `y`, and branch anchor `a`:

`population_i_y = population_i_a * (official_population_g_y / sum_i(population_i_a))`

The 2025 branch is additionally evaluated at 2019 for an anchor-transition diagnostic. The selected panel series uses the observed 2019 branch for 2019, preventing duplicate cell-year-quarter rows. Annual values are repeated for all four quarters.

In [9]:
if not MUNICIPAL_POPULATION_PATH.exists():
    raise FileNotFoundError(f"Dual-anchor population backcast requires {MUNICIPAL_POPULATION_PATH}")

municipal_population_wide = pd.read_csv(MUNICIPAL_POPULATION_PATH, sep=";", encoding="cp1252")
population_columns = [f"POP_{year}" for year in range(START_YEAR, END_YEAR + 1)]
required_columns = ["LAU_CODE", "LAU_NAME", *population_columns]
missing_columns = [column for column in required_columns if column not in municipal_population_wide.columns]
if missing_columns:
    raise ValueError(f"Missing columns in {MUNICIPAL_POPULATION_PATH.name}: {missing_columns}")

municipal_population = municipal_population_wide[required_columns].copy()
municipal_population["municipality_id"] = municipal_population["LAU_CODE"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True).str.zfill(5)
raster_ids = set(raster["municipality_id"].dropna().astype("string"))
population_ids = set(municipal_population["municipality_id"].dropna())
if raster_ids != population_ids:
    raise ValueError(f"Municipality-ID mismatch; only raster: {sorted(raster_ids - population_ids)[:10]}, only population CSV: {sorted(population_ids - raster_ids)[:10]}")

municipal_population = municipal_population.melt(
    id_vars=["municipality_id", "LAU_NAME"],
    value_vars=population_columns,
    var_name="year",
    value_name="municipality_population_year",
)
municipal_population["year"] = municipal_population["year"].str.replace("POP_", "", regex=False).astype(int)
municipal_population["municipality_population_year"] = pd.to_numeric(municipal_population["municipality_population_year"], errors="raise")


def build_anchor_branch(anchor_year: int, target_years) -> pd.DataFrame:
    anchor_column = f"population_{anchor_year}"
    municipality_anchor = (
        raster.groupby("municipality_id", as_index=False)[anchor_column]
        .sum()
        .rename(columns={anchor_column: "municipality_population_anchor_grid"})
    )
    targets = municipal_population.loc[municipal_population["year"].isin(target_years)].merge(
        municipality_anchor, on="municipality_id", how="left", validate="many_to_one"
    )
    invalid = targets["municipality_population_anchor_grid"].isna() | targets["municipality_population_anchor_grid"].le(0)
    if invalid.any():
        raise ValueError(f"{anchor_year} anchor has no positive grid population for municipalities: {sorted(targets.loc[invalid, 'municipality_id'].unique())[:10]}")
    targets["scaling_factor"] = targets["municipality_population_year"] / targets["municipality_population_anchor_grid"]

    anchor_columns = list(dict.fromkeys(["grid_id", "municipality_id", "population_2019", "population_2025", anchor_column]))
    cells = raster[anchor_columns].copy()
    branch = cells.merge(targets, on="municipality_id", how="inner", validate="many_to_many")
    branch["population_backcast"] = branch[anchor_column] * branch["scaling_factor"]
    branch["anchor_year"] = anchor_year
    branch["backcast_method"] = f"annual_municipal_scaling_from_{anchor_year}_100m_grid"
    return branch


branch_2019 = build_anchor_branch(2019, range(START_YEAR, 2020))
branch_2025 = build_anchor_branch(2025, range(2019, END_YEAR + 1))

# Keep the overlapping 2019 estimate from the 2025 branch for validation, but
# select the observed 2019 spatial branch for the panel's unique 2019 value.
anchor_transition_2019 = branch_2019.loc[branch_2019["year"].eq(2019)].merge(
    branch_2025.loc[branch_2025["year"].eq(2019), ["grid_id", "population_backcast"]].rename(
        columns={"population_backcast": "population_backcast_from_2025_anchor"}
    ),
    on="grid_id",
    how="outer",
)
population_annual = pd.concat(
    [branch_2019, branch_2025.loc[branch_2025["year"].ge(2020)]],
    ignore_index=True,
)

quarters_for_population = pd.DataFrame({"quarter": [1, 2, 3, 4]})
population_backcast = population_annual.merge(quarters_for_population, how="cross")
population_backcast = population_backcast[
    [
        "grid_id", "year", "quarter", "municipality_id", "population_backcast",
        "population_2019", "population_2025", "municipality_population_year",
        "municipality_population_anchor_grid", "scaling_factor", "anchor_year", "backcast_method",
    ]
]
if population_backcast.duplicated(["grid_id", "year", "quarter"]).any():
    raise ValueError("Dual-anchor construction produced duplicate grid-year-quarter rows")
population_backcast.to_parquet(POPULATION_BACKCAST_OUTPUT, index=False)
print(f"Saved dual-anchor population backcast: {POPULATION_BACKCAST_OUTPUT}")
annual_totals = population_backcast.drop_duplicates(["grid_id", "year"]).groupby(["year", "anchor_year"], as_index=False)["population_backcast"].sum()
print(annual_totals.to_string(index=False))

Saved dual-anchor population backcast: D:\CO2_Masterarbeit\CO2_Masterarbeit\ANAL\data\population_backcast_100m_quarterly.parquet


 year  anchor_year  population_backcast
 2015         2019            1221570.0
 2016         2019            1232012.0
 2017         2019            1237298.0
 2018         2019            1240214.0
 2019         2019            1243052.0
 2020         2025            1246395.0
 2021         2025            1247077.0
 2022         2025            1252922.0
 2023         2025            1265198.0
 2024         2025            1269801.0
 2025         2025            1271716.0


In [10]:
display(population_backcast.head())
transition_summary = pd.Series({
    "cells_in_2019_anchor": anchor_transition_2019["population_backcast"].gt(0).sum(),
    "cells_in_2025_anchor_backcast_to_2019": anchor_transition_2019["population_backcast_from_2025_anchor"].gt(0).sum(),
    "cells_positive_in_either_2019_estimate": anchor_transition_2019[["population_backcast", "population_backcast_from_2025_anchor"]].fillna(0).gt(0).any(axis=1).sum(),
    "2019_total_from_2019_anchor": anchor_transition_2019["population_backcast"].sum(),
    "2019_total_from_2025_anchor": anchor_transition_2019["population_backcast_from_2025_anchor"].sum(),
})
display(transition_summary.to_frame("value"))

,grid_id,year,quarter,municipality_id,population_backcast,population_2019,population_2025,municipality_population_year,municipality_population_anchor_grid,scaling_factor,anchor_year,backcast_method
0,AT_CRS3035RES100mN2626400E4740200,2015,1,61054,0.0,0.0,0.0,3778,3653.0,1.034218,2019,annual_municipal_scaling_from_2019_100m_grid
1,AT_CRS3035RES100mN2626400E4740200,2015,2,61054,0.0,0.0,0.0,3778,3653.0,1.034218,2019,annual_municipal_scaling_from_2019_100m_grid
2,AT_CRS3035RES100mN2626400E4740200,2015,3,61054,0.0,0.0,0.0,3778,3653.0,1.034218,2019,annual_municipal_scaling_from_2019_100m_grid
3,AT_CRS3035RES100mN2626400E4740200,2015,4,61054,0.0,0.0,0.0,3778,3653.0,1.034218,2019,annual_municipal_scaling_from_2019_100m_grid
4,AT_CRS3035RES100mN2626400E4740200,2016,1,61054,0.0,0.0,0.0,3794,3653.0,1.038598,2019,annual_municipal_scaling_from_2019_100m_grid


,value
cells_in_2019_anchor,121992.0
cells_in_2025_anchor_backcast_to_2019,123403.0
cells_positive_in_either_2019_estimate,128057.0
2019_total_from_2019_anchor,1243052.0
2019_total_from_2025_anchor,1243052.0


## 9. Build the 100 m Raster-Quarter Panel

The preliminary analytical cell set is fixed as the union of cells with positive observed population in 2019, positive observed population in 2025, or at least one firm present during the analysis window. Thus population appearance/disappearance and temporary firm presence cannot remove a cell from other panel quarters. Both observed population columns remain in the panel alongside the selected dual-anchor annual backcast.

In [11]:
analysis_start = pd.Timestamp(f"{START_YEAR}-01-01")

if population_backcast is None and POPULATION_BACKCAST_OUTPUT.exists():
    population_backcast = pd.read_parquet(POPULATION_BACKCAST_OUTPUT)

positive_population_cell_ids = raster.loc[
    raster[["population_2019", "population_2025"]].gt(0).any(axis=1),
    "grid_id",
].dropna().unique()

firm_present_in_window = (
    firms["founding_date"].notna()
    & (firms["founding_date"] <= CENSORING_DATE)
    & (firms["exit_date"].isna() | (firms["exit_date"] > analysis_start))
)
firm_cell_ids = firms.loc[firm_present_in_window, "grid_id_100m"].dropna().unique()
eligible_cell_ids = np.union1d(positive_population_cell_ids, firm_cell_ids)
panel_cells = raster[raster["grid_id"].isin(eligible_cell_ids)].copy()
panel_cells["preliminary_panel_cell"] = True

quarters = pd.period_range(f"{START_YEAR}Q1", f"{END_YEAR}Q4", freq="Q")
quarter_table = pd.DataFrame(
    {
        "period": quarters.astype(str),
        "year": quarters.year,
        "quarter": quarters.quarter,
    }
)

panel = panel_cells[["grid_id", "municipality_id", "municipality_name", "population_2019", "population_2025", "preliminary_panel_cell"]].merge(
    quarter_table,
    how="cross",
)

births = (
    firms.dropna(subset=["grid_id_100m", "founding_quarter"])
    .loc[lambda df: df["founding_date"].between(analysis_start, CENSORING_DATE, inclusive="both")]
    .groupby(["grid_id_100m", "founding_quarter"], as_index=False)
    .size()
    .rename(columns={"grid_id_100m": "grid_id", "founding_quarter": "period", "size": "births"})
)

births_by_fachgruppe = (
    firms.dropna(subset=["grid_id_100m", "founding_quarter", "Fachgruppe_ID"])
    .loc[lambda df: df["founding_date"].between(analysis_start, CENSORING_DATE, inclusive="both")]
    .groupby(["grid_id_100m", "founding_quarter", "Fachgruppe_ID"], as_index=False)
    .size()
    .rename(columns={"grid_id_100m": "grid_id", "founding_quarter": "period", "size": "births"})
)
births_by_fachgruppe.to_parquet(BIRTHS_BY_FACHGRUPPE_OUTPUT, index=False)
print(f"Saved births by Fachgruppe: {BIRTHS_BY_FACHGRUPPE_OUTPUT}")

exits = (
    firms.dropna(subset=["grid_id_100m", "exit_quarter"])
    .loc[lambda df: df["exit_date"].between(analysis_start, CENSORING_DATE, inclusive="both")]
    .groupby(["grid_id_100m", "exit_quarter"], as_index=False)
    .size()
    .rename(columns={"grid_id_100m": "grid_id", "exit_quarter": "period", "size": "exits"})
)

panel = panel.merge(births, on=["grid_id", "period"], how="left")
panel = panel.merge(exits, on=["grid_id", "period"], how="left")
panel["births"] = panel["births"].fillna(0).astype(int)
panel["exits"] = panel["exits"].fillna(0).astype(int)

# End-of-quarter stocks are descriptive only. For the H2 regressors use the
# lagged columns below: founders in quarter t must not enter their own exposure.
firm_dates = firms.dropna(subset=["grid_id_100m", "founding_date"])[["firm_id", "grid_id_100m", "founding_date", "exit_date", "Fachgruppe_ID"]].copy()
firm_dates["exit_date_filled"] = firm_dates["exit_date"].fillna(pd.Timestamp.max)
firm_dates["Fachgruppe_ID"] = firm_dates["Fachgruppe_ID"].astype("string")
fachgruppe_t_columns = [f"fachgruppe_{fachgruppe_id}_active_firms_t" for fachgruppe_id in FACHGRUPPE_IDS]
fachgruppe_tminus1_columns = [f"fachgruppe_{fachgruppe_id}_active_firms_tminus1" for fachgruppe_id in FACHGRUPPE_IDS]

active_records = []
for period in quarters:
    quarter_end = period.end_time.normalize()
    previous_quarter_end = (period - 1).end_time.normalize()
    active_now = firm_dates[(firm_dates["founding_date"] <= quarter_end) & (firm_dates["exit_date_filled"] > quarter_end)]
    active_previous = firm_dates[(firm_dates["founding_date"] <= previous_quarter_end) & (firm_dates["exit_date_filled"] > previous_quarter_end)]

    active_now_counts = active_now.groupby("grid_id_100m").size().rename("active_firms_t")
    active_previous_counts = active_previous.groupby("grid_id_100m").size().rename("active_firms_tminus1")
    active_now_sector_counts = (
        active_now.dropna(subset=["Fachgruppe_ID"])
        .groupby(["grid_id_100m", "Fachgruppe_ID"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=FACHGRUPPE_IDS, fill_value=0)
    )
    active_now_sector_counts.columns = fachgruppe_t_columns
    active_previous_sector_counts = (
        active_previous.dropna(subset=["Fachgruppe_ID"])
        .groupby(["grid_id_100m", "Fachgruppe_ID"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=FACHGRUPPE_IDS, fill_value=0)
    )
    active_previous_sector_counts.columns = fachgruppe_tminus1_columns
    active_counts = pd.concat(
        [
            active_now_counts,
            active_previous_counts,
            active_now_sector_counts,
            active_previous_sector_counts,
        ],
        axis=1,
    ).fillna(0).astype(int).reset_index()
    active_counts = active_counts.rename(columns={"grid_id_100m": "grid_id"})
    active_counts["period"] = str(period)
    active_records.append(active_counts)

active_panel = pd.concat(active_records, ignore_index=True)
panel = panel.merge(active_panel, on=["grid_id", "period"], how="left")
panel["active_firms_t"] = panel["active_firms_t"].fillna(0).astype(int)
panel["active_firms_tminus1"] = panel["active_firms_tminus1"].fillna(0).astype(int)
panel[fachgruppe_t_columns + fachgruppe_tminus1_columns] = panel[fachgruppe_t_columns + fachgruppe_tminus1_columns].fillna(0).astype(int)

if population_backcast is not None:
    panel = panel.merge(
        population_backcast[["grid_id", "year", "quarter", "population_backcast"]],
        on=["grid_id", "year", "quarter"],
        how="left",
    )
else:
    panel["population_backcast"] = pd.NA

panel.to_parquet(PANEL_OUTPUT, index=False)
print(f"Saved raster-quarter panel: {PANEL_OUTPUT}")
print(f"Panel cells: {panel_cells['grid_id'].nunique():,}")
print(f"Panel rows: {len(panel):,}")

Saved births by Fachgruppe: D:\CO2_Masterarbeit\CO2_Masterarbeit\ANAL\data\births_by_fachgruppe_100m.parquet


Saved raster-quarter panel: D:\CO2_Masterarbeit\CO2_Masterarbeit\ANAL\data\raster_quarter_panel_100m.parquet
Panel cells: 130,092
Panel rows: 5,724,048


## 10. Validation Summary

In [12]:
print("Raster checks")
check(raster.crs.to_epsg() == 3035, "raster CRS is EPSG:3035")
check(raster["grid_id"].is_unique, "grid_id is unique")
check(raster["municipality_id"].notna().all(), "all raster cells have municipality_id")
check((raster["resolution_m"] == 100).all(), "all cells are marked as 100 m")
for year, layer in population_grids.items():
    check(np.isclose(raster[f"population_{year}"].sum(), layer["population"].sum()), f"{year} population sum matches Styria-filtered population grid")
check(raster[["population_2019", "population_2025"]].gt(0).any(axis=1).sum() == len(set(population_2019["cell_id"]) | set(population_2025["cell_id"])), "raster contains the union of populated cells from both anchors")

print("\nFirm checks")
check(len(firms_source) > 0, "source firm input has rows")
check(len(firms) == len(firms_source), "analytical firm output has the same number of rows as the input")
check(firms["grid_id_100m"].notna().all(), "all firms receive grid_id_100m")
firm_fachgruppe_ids = set(firms["Fachgruppe_ID"].dropna().astype("string").unique())
check(firm_fachgruppe_ids == set(FACHGRUPPE_IDS), "firm Fachgruppe_ID values match the expected 95 Fachorganisationen")
invalid_exit_dates = firms["exit_date"].notna() & (firms["exit_date"] < firms["founding_date"])
check(not invalid_exit_dates.any(), "no exit date before founding date")

print("\nPopulation backcast checks")
validation = (
    population_backcast.drop_duplicates(["grid_id", "year"])
    .groupby(["municipality_id", "year", "anchor_year"], as_index=False)
    .agg(
        raster_sum=("population_backcast", "sum"),
        official_population=("municipality_population_year", "first"),
        scaling_factor=("scaling_factor", "first"),
    )
)
validation["absolute_deviation"] = validation["raster_sum"] - validation["official_population"]
validation["relative_deviation"] = validation["absolute_deviation"] / validation["official_population"]
display(validation.sort_values("absolute_deviation", key=lambda s: s.abs(), ascending=False).head(10))
check(np.allclose(validation["raster_sum"], validation["official_population"]), "every dual-anchor municipality-year raster sum matches its official municipality total")
check(population_backcast.loc[population_backcast["year"].le(2019), "anchor_year"].eq(2019).all(), "2015-2019 use the 2019 spatial anchor")
check(population_backcast.loc[population_backcast["year"].ge(2020), "anchor_year"].eq(2025).all(), "2020-2025 use the 2025 spatial anchor")
check(not population_backcast.duplicated(["grid_id", "year", "quarter"]).any(), "population backcast has unique grid-year-quarter rows")
suspicious = validation[(validation["scaling_factor"] < 0.75) | (validation["scaling_factor"] > 1.25)]
print(f"Suspicious scaling factors outside 0.75-1.25: {len(suspicious):,}")

print("\nPanel checks")
panel_cell_ids = set(panel["grid_id"].unique())
expected_panel_cell_ids = set(positive_population_cell_ids) | set(firm_cell_ids)
check(panel_cell_ids == expected_panel_cell_ids, "panel cell set is the union of cells populated in either 2019 or 2025 and cells with a firm present in the analysis window")
check(set(raster.loc[raster["population_2019"].gt(0), "grid_id"]).issubset(panel_cell_ids), "every cell populated in 2019 remains in the panel")
check(set(raster.loc[raster["population_2025"].gt(0), "grid_id"]).issubset(panel_cell_ids), "every cell populated in 2025 remains in the panel")
check({"population_2019", "population_2025", "population_backcast"}.issubset(panel.columns), "panel retains both observed anchors and the selected backcast")
expected_rows = panel["grid_id"].nunique() * len(quarters)
check(len(panel) == expected_rows, "panel has one row per preliminary panel cell and quarter")
check(panel["period"].min() == f"{START_YEAR}Q1" and panel["period"].max() == f"{END_YEAR}Q4", "panel covers 2015Q1-2025Q4")
expected_births = firms[firms["founding_date"].between(pd.Timestamp(f"{START_YEAR}-01-01"), CENSORING_DATE, inclusive="both")].shape[0]
expected_exits = firms[firms["exit_date"].between(pd.Timestamp(f"{START_YEAR}-01-01"), CENSORING_DATE, inclusive="both")].shape[0]
check(panel["births"].sum() == expected_births, "panel births match firm events in period")
births_without_fachgruppe = firms.loc[
    firms["founding_date"].between(pd.Timestamp(f"{START_YEAR}-01-01"), CENSORING_DATE, inclusive="both")
    & firms["Fachgruppe_ID"].isna(),
].shape[0]
print(f"Births without Fachgruppe_ID: {births_without_fachgruppe:,}")
check(
    births_by_fachgruppe["births"].sum() == panel["births"].sum() - births_without_fachgruppe,
    "Fachgruppe births match panel births minus births without Fachgruppe_ID",
)
check(panel["exits"].sum() == expected_exits, "panel exits match firm events in period")
check((panel["active_firms_tminus1"] >= 0).all(), "active_firms_tminus1 is never negative")
fachgruppe_t_columns = [f"fachgruppe_{fachgruppe_id}_active_firms_t" for fachgruppe_id in FACHGRUPPE_IDS]
fachgruppe_tminus1_columns = [f"fachgruppe_{fachgruppe_id}_active_firms_tminus1" for fachgruppe_id in FACHGRUPPE_IDS]
check(set(fachgruppe_t_columns + fachgruppe_tminus1_columns).issubset(panel.columns), "panel has the Fachgruppe stock columns")
if set(fachgruppe_t_columns + fachgruppe_tminus1_columns).issubset(panel.columns):
    fachgruppe_sum_t = panel[fachgruppe_t_columns].sum(axis=1)
    fachgruppe_sum_tminus1 = panel[fachgruppe_tminus1_columns].sum(axis=1)
    check((fachgruppe_sum_t == panel["active_firms_t"]).all(), "Fachgruppe t stocks sum to active_firms_t")
    check((fachgruppe_sum_tminus1 == panel["active_firms_tminus1"]).all(), "Fachgruppe lagged stocks sum to active_firms_tminus1")

Raster checks
OK: raster CRS is EPSG:3035


OK: grid_id is unique
OK: all raster cells have municipality_id
OK: all cells are marked as 100 m
OK: 2019 population sum matches Styria-filtered population grid
OK: 2025 population sum matches Styria-filtered population grid


OK: raster contains the union of populated cells from both anchors

Firm checks
OK: source firm input has rows
OK: analytical firm output has the same number of rows as the input
OK: all firms receive grid_id_100m
OK: firm Fachgruppe_ID values match the expected 95 Fachorganisationen
OK: no exit date before founding date

Population backcast checks


,municipality_id,year,anchor_year,raster_sum,official_population,scaling_factor,absolute_deviation,relative_deviation
938,61108,2018,2019,24645.0,24645,1.003951,-3.637979e-12,-1.476153e-16
935,61108,2015,2019,24695.0,24695,1.005988,-3.637979e-12,-1.473164e-16
936,61108,2016,2019,25350.0,25350,1.032671,3.637979e-12,1.435100e-16
2376,62140,2015,2019,23188.0,23188,1.022714,3.637979e-12,1.568906e-16
2377,62140,2016,2019,23067.0,23067,1.017377,-3.637979e-12,-1.577136e-16
2386,62140,2025,2025,21907.0,21907,1.003895,3.637979e-12,1.660647e-16
1058,61120,2017,2019,11143.0,11143,1.000539,1.818989e-12,1.632405e-16
156,60350,2017,2019,8650.0,8650,1.001273,-1.818989e-12,-2.102878e-16
2180,62041,2017,2019,12658.0,12658,1.002852,-1.818989e-12,-1.437027e-16
1057,61120,2016,2019,11227.0,11227,1.008081,-1.818989e-12,-1.620192e-16


OK: every dual-anchor municipality-year raster sum matches its official municipality total


...

OK: population backcast has unique grid-year-quarter rows
Suspicious scaling factors outside 0.75-1.25: 0

Panel checks


OK: panel cell set is the union of cells populated in either 2019 or 2025 and cells with a firm present in the analysis window
OK: every cell populated in 2019 remains in the panel
OK: every cell populated in 2025 remains in the panel
OK: panel retains both observed anchors and the selected backcast


OK: panel has one row per preliminary panel cell and quarter


OK: panel covers 2015Q1-2025Q4
OK: panel births match firm events in period
Births without Fachgruppe_ID: 0
OK: Fachgruppe births match panel births minus births without Fachgruppe_ID
OK: panel exits match firm events in period
OK: active_firms_tminus1 is never negative
OK: panel has the Fachgruppe stock columns


OK: Fachgruppe t stocks sum to active_firms_t
OK: Fachgruppe lagged stocks sum to active_firms_tminus1
